# Assignment 06: RNN Foundations in PyTorch


**Student Name:** Elvia Aptanisa - Exchange Student

**Student ID:** S26076566

---

## Part A: Manual RNN Calculation

Final Result


*   h1 : [0.4621 , 0.0997]
*   h2 : [-0.0973 , 0.2925]
*   h3 : [0.1094 , -0.1526]
*   h4 : [0.2973 , 0.2969]



## Part B: Reproduce the Same RNN in PyTorch

Setup

In [1]:
import torch
import torch.nn as nn

B1. Create the input tensor

In [4]:
x = torch.tensor([[
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
    [1.0, 1.0, 0.0]
]], dtype=torch.float32)

print("x.shape          :", x.shape)

x.shape          : torch.Size([1, 4, 3])


B2. Create the RNN model (same as manual calculation)



In [5]:
rnn = nn.RNN(input_size=3, hidden_size=2, batch_first=True)

output, h_n = rnn(x)

print("output.shape     :", output.shape)
print("h_n.shape        :", h_n.shape)
print("\noutput tensor:\n", output)
print("\nh_n tensor:\n", h_n)

output.shape     : torch.Size([1, 4, 2])
h_n.shape        : torch.Size([1, 1, 2])

output tensor:
 tensor([[[-0.2565,  0.4148],
         [ 0.3163, -0.1436],
         [ 0.1628, -0.4800],
         [ 0.4727,  0.4593]]], grad_fn=<TransposeBackward1>)

h_n tensor:
 tensor([[[0.4727, 0.4593]]], grad_fn=<StackBackward0>)


**Answer:**

1. output[:, -1, :] represents the hidden state after the last time step (final output).
2. h_n[-1] represents the final hidden state of the last layer.
3. They are the same for one-layer RNN because h_n stores the hidden state after the last time step.
4. If batch_first=False, the input and output shapes will swap: sequence length comes first.

---
## Part C: DNA Sequence Classifier

Setup

In [6]:
import torch
import torch.nn as nn
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader

C1. Generate Data

In [12]:
random.seed(42)
np.random.seed(42)

MOTIF = "ATG"

def generate_dna_sequence(length):
    return ''.join(random.choice('ACGT') for _ in range(length))

def generate_dataset(num_samples, motif="ATG"):
    sequences = []
    labels = []

    for _ in range(num_samples):
        length = random.randint(30, 80)

        if random.random() < 0.5:
            seq = generate_dna_sequence(length)
            pos = random.randint(0, length - len(motif))
            seq = seq[:pos] + motif + seq[pos + len(motif):]
            label = 1
        else:
            seq = generate_dna_sequence(length)
            while motif in seq:
                seq = generate_dna_sequence(length)
            label = 0

        sequences.append(seq)
        labels.append(label)

    return sequences, labels

train_seq, train_labels = generate_dataset(2000)
val_seq,   val_labels   = generate_dataset(500)
test_seq,  test_labels  = generate_dataset(500)

print("C1 - Data generated successfully!")
print(f"Train samples     : {len(train_seq)}")
print(f"Validation samples: {len(val_seq)}")
print(f"Test samples      : {len(test_seq)}")
print(f"Train positive ratio: {sum(train_labels)/len(train_labels):.3f}")

print("\nExample sequence:")
print("Seq  :", train_seq[0])
print("Label:", train_labels[0], "(1 = has ATG, 0 = no ATG)")

C1 - Data generated successfully!
Train samples     : 2000
Validation samples: 500
Test samples      : 500
Train positive ratio: 0.491

Example sequence:
Seq  : GCCCAATAAACCACTCTGACTGGCCGAATAGGGATATAGGCAACGACATGTATGGCGACCCTTGCGACAG
Label: 1 (1 = has ATG, 0 = no ATG)


C2. Encode and Pad

In [13]:
char2idx = {"A": 0, "C": 1, "G": 2, "T": 3, "<PAD>": 4}
idx2char = {v: k for k, v in char2idx.items()}

class DNADataset(Dataset):
    def __init__(self, sequences, labels, char2idx):
        self.sequences = sequences
        self.labels = labels
        self.char2idx = char2idx

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        encoded = [self.char2idx[c] for c in seq]
        return torch.tensor(encoded), torch.tensor(label)

    def collate_fn(self, batch):
        sequences = [item[0] for item in batch]
        labels = [item[1] for item in batch]

        lengths = torch.tensor([len(seq) for seq in sequences])

        padded = nn.utils.rnn.pad_sequence(sequences,
                                           batch_first=True,
                                           padding_value=self.char2idx["<PAD>"])

        labels = torch.stack(labels)

        return padded, lengths, labels


train_dataset = DNADataset(train_seq, train_labels, char2idx)
val_dataset   = DNADataset(val_seq,   val_labels,   char2idx)
test_dataset  = DNADataset(test_seq,  test_labels,  char2idx)

print("C2 - Datasets created successfully!")
print("Vocabulary size :", len(char2idx))
print("PAD index       :", char2idx["<PAD>"])

x_batch, lengths, y_batch = train_dataset.collate_fn([train_dataset[0]])
print("\nExample after encoding & padding:")
print("Padded shape :", x_batch.shape)
print("Lengths      :", lengths.item())
print("Label        :", y_batch.item())

C2 - Datasets created successfully!
Vocabulary size : 5
PAD index       : 4

Example after encoding & padding:
Padded shape : torch.Size([1, 70])
Lengths      : 70
Label        : 1


C3. Train the Classifier

In [18]:
class DNAClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(),
                                                   batch_first=True, enforce_sorted=False)
        packed_output, (h_n, c_n) = self.rnn(packed)
        last_hidden = h_n[-1]
        return self.fc(last_hidden)

VOCAB_SIZE = len(char2idx)
EMBED_DIM = 32
HIDDEN_SIZE = 64
NUM_CLASSES = 2
PAD_IDX = char2idx["<PAD>"]
BATCH_SIZE = 64
NUM_EPOCHS = 30
LEARNING_RATE = 0.001

model = DNAClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, NUM_CLASSES, PAD_IDX)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=train_dataset.collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=val_dataset.collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=test_dataset.collate_fn)

print("C3 - LSTM Model created!")

def evaluate_accuracy(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, lengths, y_batch in data_loader:
            outputs = model(x_batch, lengths)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    return correct / total

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        for x_batch, lengths, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(x_batch, lengths)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            val_acc = evaluate_accuracy(model, val_loader)
            print(f"Epoch [{epoch+1:2d}/{num_epochs}]  Loss: {avg_loss:.4f}   Val Acc: {val_acc:.4f}")

    test_acc = evaluate_accuracy(model, test_loader)
    print(f"\nFinal Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
    return test_acc

print("\nStarting LSTM training...\n")
final_test_acc = train_model(model, train_loader, val_loader, criterion, optimizer, NUM_EPOCHS)

C3 - LSTM Model created!

Starting LSTM training...

Epoch [ 1/30]  Loss: 0.6944   Val Acc: 0.5580
Epoch [ 5/30]  Loss: 0.6607   Val Acc: 0.5740
Epoch [10/30]  Loss: 0.0096   Val Acc: 1.0000
Epoch [15/30]  Loss: 0.0025   Val Acc: 1.0000
Epoch [20/30]  Loss: 0.0012   Val Acc: 1.0000
Epoch [25/30]  Loss: 0.0007   Val Acc: 1.0000
Epoch [30/30]  Loss: 0.0005   Val Acc: 1.0000

Final Test Accuracy: 1.0000 (100.00%)


C4. Generalization Tests

In [19]:
def predict_sequence(model, seq_str):
    model.eval()
    encoded = [char2idx[c] for c in seq_str]
    x = torch.tensor([encoded])
    lengths = torch.tensor([len(encoded)])

    with torch.no_grad():
        output = model(x, lengths)
        prob = torch.softmax(output, dim=1)[0]
        pred = torch.argmax(prob).item()
        confidence = prob[pred].item()

    return pred, confidence, prob[1].item()

test_cases = [
    ("Motif at beginning", "ATG" + "CGTA" * 15),
    ("Motif in middle",    "CGTA" * 8 + "ATG" + "CGTA" * 8),
    ("Motif at the end",   "CGTA" * 15 + "ATG"),
    ("Long sequence",      "CGTA" * 5 + "ATG" + "CGTA" * 20),
    ("Noisy repeated",     "AAAATGCCCC" + "G" * 30 + "T" * 10 + "ATG")
]

print("C4 - Generalization Test Results:\n")
print(f"{'Test Case':<25} {'True Label':<12} {'Pred Label':<12} {'Prob of 1':<10} {'Confidence':<10}")
print("-" * 75)

for name, seq in test_cases:
    true_label = 1 if MOTIF in seq else 0
    pred, conf, prob1 = predict_sequence(model, seq)
    print(f"{name:<25} {true_label:<12} {pred:<12} {prob1:.4f}     {conf:.4f}")

    display_seq = seq if len(seq) <= 60 else seq[:30] + "..." + seq[-20:]
    print(f"Sequence: {display_seq}\n")

C4 - Generalization Test Results:

Test Case                 True Label   Pred Label   Prob of 1  Confidence
---------------------------------------------------------------------------
Motif at beginning        1            1            1.0000     1.0000
Sequence: ATGCGTACGTACGTACGTACGTACGTACGT...CGTACGTACGTACGTACGTA

Motif in middle           1            1            1.0000     1.0000
Sequence: CGTACGTACGTACGTACGTACGTACGTACG...CGTACGTACGTACGTACGTA

Motif at the end          1            1            0.9724     0.9724
Sequence: CGTACGTACGTACGTACGTACGTACGTACG...ACGTACGTACGTACGTAATG

Long sequence             1            1            1.0000     1.0000
Sequence: CGTACGTACGTACGTACGTAATGCGTACGT...CGTACGTACGTACGTACGTA

Noisy repeated            1            1            0.9999     0.9999
Sequence: AAAATGCCCCGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGTTTTTTTTTTATG



---
## Follow-up Questions

**Follow-Up Questions**


1. Why do we use nn.Embedding before the RNN?
We use nn.Embedding to convert the DNA letters (A, C, G, T) into dense vectors.
The RNN cannot understand characters directly, so embedding turns each letter into a meaningful number vector that the RNN can learn from.
2. Why is this a many-to-one task?
Because the model reads many inputs (all nucleotides in the sequence) but produces only one output (the final label: 1 or 0).
It is “many inputs → one final decision”.
3. Why do we use the final hidden state for classification?
The final hidden state (h_n[-1]) contains the summary of the entire sequence.
It keeps the most important information after the RNN has read all the DNA letters from beginning to end.
4. Why do we pad sequences in a batch?
Sequences have different lengths. PyTorch needs all sequences in one batch to have the same length.
We pad the shorter ones with <PAD> tokens so we can process them together efficiently.
5. Why does pack_padded_sequence help when sequences have different lengths?
It tells the RNN to ignore the padding tokens.
Without packing, the RNN would treat padding as real DNA letters and the hidden state would be polluted by useless padding.
6. What kind of information might a vanilla RNN forget on long sequences?
A vanilla RNN can easily forget important information from the beginning of a long sequence (this is called the vanishing gradient problem).
7. What problem are LSTMs and GRUs designed to reduce compared with a vanilla RNN?
They are designed to reduce the vanishing gradient problem.
LSTMs and GRUs can remember important information for much longer sequences.
8. What do gates help an LSTM or GRU decide while reading a sequence?
Gates help the model decide:
What information to forget
What new information to store
What information to output
This makes the model much smarter at keeping only useful memory.

9. If you replaced nn.RNN with nn.LSTM, what extra value does PyTorch return?
LSTM returns two things: h_n (hidden state) and c_n (cell state).
The cell state is extra memory that helps LSTM remember information over long sequences.